# Does the Vina score separate ACE actives from inactives? (docking screening-power test)

Redocking showed that Vina cannot reliably reproduce the lisinopril pose with the flexible protocol. That does **not** automatically mean the score is useless for *sorting compounds*, which is what the pipeline needs from it. This notebook tests exactly that:

1. Pick **measured actives** (IC50 <= the activity cutoff) and **measured inactives** (IC50 >= 10,000 nM) from the cleaned ChEMBL training set, biochemical assays only, prodrugs removed, at most a few compounds per scaffold.
2. By default the inactives are **matched to the actives on heavy-atom count**. Vina scores grow with ligand size, and potent ACE inhibitors tend to be large, so without matching, "size" would masquerade as "binding".
3. Dock everything with **the same protocol that is used for the candidates**: the validated conformer-ensemble protocol (notebook 5b) when `1o86_redocking_validated_settings.json` exists, otherwise the flexible Vina protocol of notebook 5. The protocol name is stored in the result so notebook 9 only applies the verdict to docking done with the same protocol.
4. Report the **ROC AUC of the Vina score with a bootstrap 95% CI**, next to four reference scores: heavy-atom count, ligand efficiency, a **size-adjusted Vina score** and presence of a zinc-binding group.

The verdict is fixed in advance: `USEFUL_BEYOND_SIZE` only if the lower CI bound of the Vina AUC is >= 0.60 **and** the size-adjusted AUC lower bound is > 0.50. With about 30 + 30 compounds the confidence interval is wide (roughly +/- 0.12 around an AUC of 0.75, by my approximation; the bootstrap printed below is the real figure), so a clearly positive result is needed before docking is counted as evidence. The result is saved and read by notebook 9.

Ensemble docking is slow (about 100 conformers per compound), so in that mode the default is 20 + 20 compounds and one seed, which widens the confidence interval further; edit `N_ACTIVES`, `N_INACTIVES` and `SP_SEEDS` to trade time for power. This test uses ChEMBL compounds that the ML models were trained on; that is irrelevant here, because only the docking score is being evaluated.

In [ ]:
from pathlib import Path
import os, sys, json, warnings
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ROOT = Path('/content/drive/MyDrive/ACE_Workflow')
except ImportError:
    ROOT = Path(os.environ.get('ACE_WORKFLOW_ROOT', Path.cwd())).resolve()
if not (ROOT / 'ace_config.py').exists():
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ace_config.py').exists()), ROOT)
if not (ROOT / 'ace_config.py').exists():
    raise FileNotFoundError('Could not find ace_config.py. Mount Drive or set ACE_WORKFLOW_ROOT.')
os.environ['ACE_WORKFLOW_ROOT'] = str(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import ace_config

import hashlib, shutil, subprocess, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score, roc_curve
RDLogger.DisableLog('rdApp.*')

PATHS = ace_config.PATHS; RANDOM_STATE = ace_config.RANDOM_STATE; DOCKING_CPU = ace_config.CPU_WORKERS
PROCESSED, DOCK_REPORTS, FIG = PATHS['DATA_PROCESSED'], PATHS['DOCKING_REPORTS'], PATHS['VIS_FIGURES']
BASE = PATHS['DOCKING'] / 'screening_power'
for d in (BASE / 'ligands', BASE / 'runs', DOCK_REPORTS, Path(FIG)): Path(d).mkdir(parents=True, exist_ok=True)

# --- design (fixed before looking at any docking result) ---
N_ACTIVES, N_INACTIVES = 30, 30
ACTIVE_MAX_NM = ace_config.IC50_CUTOFF_NM
INACTIVE_MIN_NM = ace_config.CENSORED_INACTIVE_MIN_NM
SIZE_MATCHED = True              # match inactives to actives on heavy-atom count (recommended)
SIZE_TOLERANCE = 3               # heavy atoms; widened to 2x if no match is found
MAX_PER_SCAFFOLD = 2
N_BOOTSTRAP = 2000
USEFUL_AUC_LOWER_BOUND = 0.60

VINA = shutil.which('vina') or shutil.which('vina.exe'); MEEKO_LIGAND = shutil.which('mk_prepare_ligand.py')
if not VINA or not MEEKO_LIGAND: raise RuntimeError('AutoDock Vina (>= 1.2) and Meeko must be on PATH.')
receptor_pdbqt = PATHS['REDOCKING_RECEPTOR'] / '1o86_receptor.pdbqt'
box = json.loads((PATHS['REDOCKING_ANALYSIS'] / '1o86_docking_box.json').read_text())
for f in (receptor_pdbqt,):
    if not f.exists(): raise FileNotFoundError(f'{f} not found; run notebook 4 first (receptor preparation).')

# --- docking protocol: identical to the candidate docking ---
(ROOT / 'ace_ensemble_dock.py').write_text('"""Conformer-ensemble rigid docking followed by flexible local refinement (the protocol validated in notebook 4).\n\nKept as an importable module so notebooks 4-style redocking, candidate docking (5b) and the screening-power test (7)\nall run exactly the same code.\n"""\nimport re\nimport shutil\nimport subprocess\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom rdkit import Chem\nfrom rdkit.Chem import AllChem, rdDistGeom\nfrom scipy.spatial.distance import cdist\n\nPROTOCOL_NAME = \'conformer_ensemble_rigid_docking_then_flexible_refinement\'\n\n\ndef _lines(path, first_model_only=True):\n    out, started = [], False\n    for line in Path(path).read_text(errors=\'replace\').splitlines():\n        if line.startswith(\'MODEL\'):\n            started = True\n            continue\n        if started and line.startswith(\'ENDMDL\') and first_model_only:\n            break\n        if line.startswith((\'ATOM\', \'HETATM\')):\n            out.append(line)\n    return out\n\n\ndef _heavy(line):\n    return line[77:79].strip().upper() not in {\'H\', \'HD\', \'HS\'}\n\n\ndef _xyz(line):\n    return [float(line[30:38]), float(line[38:46]), float(line[46:54])]\n\n\ndef _to_template_order(input_lines, pose_lines, conf_heavy_xyz):\n    """Map PDBQT heavy atoms (Meeko order) onto the SMILES atom order via the INPUT conformer coordinates."""\n    inp = np.array([_xyz(l) for l in input_lines if _heavy(l)])\n    pose = np.array([_xyz(l) for l in pose_lines if _heavy(l)])\n    d = cdist(inp, conf_heavy_xyz)\n    j = d.argmin(axis=1)\n    if d.min(axis=1).max() > 0.05 or len(set(j)) != len(j):\n        raise RuntimeError(\'Atom mapping between Meeko PDBQT and RDKit conformer failed.\')\n    out = np.empty_like(pose)\n    out[j] = pose\n    return out\n\n\nclass Reference:\n    """Crystal reference for a symmetry-aware heavy-atom RMSD (no superposition)."""\n\n    def __init__(self, template_mol, crystal_mol):\n        self.ref_xyz = crystal_mol.GetConformer().GetPositions()\n        self.matches = template_mol.GetSubstructMatches(crystal_mol, uniquify=False, useChirality=False, maxMatches=20000)\n        if not self.matches:\n            raise RuntimeError(\'Could not map the SMILES-based molecule onto the crystal ligand.\')\n        probe = np.zeros_like(self.ref_xyz)\n        for i, j in enumerate(self.matches[0]):\n            probe[j] = self.ref_xyz[i]\n        assert self.rmsd(probe) < 1e-6, \'RMSD mapping self-check failed\'\n\n    def rmsd(self, pose_xyz_template_order):\n        return min(float(np.sqrt(np.mean(np.sum((pose_xyz_template_order[list(m)] - self.ref_xyz) ** 2, axis=1)))) for m in self.matches)\n\n\nclass EnsembleDocker:\n    def __init__(self, receptor_pdbqt, box, vina, meeko_ligand, workdir, cpu=2):\n        self.receptor, self.box, self.vina, self.meeko = str(receptor_pdbqt), box, vina, meeko_ligand\n        self.workdir, self.cpu = Path(workdir), cpu\n        self.workdir.mkdir(parents=True, exist_ok=True)\n\n    def _box_args(self, pad=0.0):\n        b = self.box\n        return [\'--center_x\', str(b[\'center_x\']), \'--center_y\', str(b[\'center_y\']), \'--center_z\', str(b[\'center_z\']),\n                \'--size_x\', str(b[\'size_x\'] + pad), \'--size_y\', str(b[\'size_y\'] + pad), \'--size_z\', str(b[\'size_z\'] + pad)]\n\n    def _vina(self, ligand, extra, pad=0.0):\n        r = subprocess.run([self.vina, \'--receptor\', self.receptor, \'--ligand\', str(ligand), *self._box_args(pad), \'--cpu\', str(self.cpu), *extra],\n                           text=True, capture_output=True)\n        if r.returncode:\n            raise RuntimeError((r.stderr or r.stdout)[-800:])\n        return r.stdout\n\n    def dock(self, smiles, tag, seed, n_conformers=100, rigid_exhaustiveness=4, refine_top_k=20, reference=None, keep_intermediate=False):\n        """Return a dict with the best refined score/pose; a \'refined\' DataFrame lists the refined conformers.\n\n        Undefined stereocentres are embedded in both configurations (the ensemble contains both), so the reported score is the\n        best over the possible stereoisomers.\n        """\n        template = Chem.MolFromSmiles(smiles)\n        if template is None:\n            return {\'status\': \'INVALID_SMILES\', \'tag\': tag, \'seed\': seed}\n        mol_h = Chem.AddHs(template)\n        params = rdDistGeom.ETKDGv3()\n        params.randomSeed = int(seed)\n        params.pruneRmsThresh = 0.5\n        params.numThreads = self.cpu\n        cids = list(rdDistGeom.EmbedMultipleConfs(mol_h, n_conformers, params))\n        if not cids:\n            params.useRandomCoords = True\n            cids = list(rdDistGeom.EmbedMultipleConfs(mol_h, n_conformers, params))\n        if not cids:\n            return {\'status\': \'FAILED_EMBED\', \'tag\': tag, \'seed\': seed}\n        try:\n            AllChem.MMFFOptimizeMoleculeConfs(mol_h, maxIters=1000, numThreads=self.cpu)\n        except Exception:\n            pass\n        n_heavy = template.GetNumHeavyAtoms()\n        d = self.workdir / f\'{tag}_s{seed}\'\n        d.mkdir(parents=True, exist_ok=True)\n\n        records, store = [], {}\n        for cid in cids:\n            name = f\'c{cid:04d}\'\n            sdf, flex = d / f\'{name}.sdf\', d / f\'{name}.pdbqt\'\n            sdf.write_text(Chem.MolToMolBlock(mol_h, confId=cid))\n            r = subprocess.run([self.meeko, \'-i\', str(sdf), \'-o\', str(flex)], text=True, capture_output=True)\n            if r.returncode or not flex.exists():\n                continue\n            flex_text = flex.read_text().splitlines()\n            flex_atoms = _lines(flex, first_model_only=False)\n            rigid, out = d / f\'{name}_rigid.pdbqt\', d / f\'{name}_rigid_docked.pdbqt\'\n            rigid.write_text(\'\\n\'.join([l for l in flex_text if l.startswith(\'REMARK\')] + [\'ROOT\', *flex_atoms, \'ENDROOT\', \'TORSDOF 0\']) + \'\\n\')\n            try:\n                self._vina(rigid, [\'--exhaustiveness\', str(rigid_exhaustiveness), \'--num_modes\', \'3\', \'--seed\', str(seed), \'--out\', str(out)])\n            except RuntimeError:\n                continue\n            pose_lines = _lines(out)\n            score = next((float(x.split()[3]) for x in out.read_text().splitlines() if x.startswith(\'REMARK VINA RESULT\')), np.nan)\n            conf_heavy = mol_h.GetConformer(cid).GetPositions()[:n_heavy]\n            rec = {\'conformer\': cid, \'rigid_score\': score}\n            if reference is not None:\n                rec[\'rigid_rmsd_A\'] = reference.rmsd(_to_template_order(flex_atoms, pose_lines, conf_heavy))\n            records.append(rec)\n            store[cid] = {\'flex\': flex, \'pose\': pose_lines, \'atoms\': flex_atoms, \'conf_heavy\': conf_heavy}\n        if not records:\n            shutil.rmtree(d, ignore_errors=True)\n            return {\'status\': \'FAILED_DOCKING\', \'tag\': tag, \'seed\': seed}\n        ens = pd.DataFrame(records).sort_values(\'rigid_score\').reset_index(drop=True)\n\n        refined_rows = []\n        for row in ens.head(refine_top_k).itertuples():\n            o = store[row.conformer]\n            it, start_lines = iter(o[\'pose\']), []\n            for l in Path(o[\'flex\']).read_text().splitlines():\n                if l.startswith((\'ATOM\', \'HETATM\')):\n                    p = next(it)\n                    l = l[:30] + p[30:54] + l[54:]          # same atoms, rigid-pose coordinates\n                start_lines.append(l)\n            start = d / f\'c{row.conformer:04d}_start.pdbqt\'\n            start.write_text(\'\\n\'.join(start_lines) + \'\\n\')\n            refined = d / f\'c{row.conformer:04d}_refined.pdbqt\'\n            try:\n                txt = self._vina(start, [\'--local_only\', \'--out\', str(refined)], pad=12.0)   # larger box: edge poses may poke out of the docking box\n            except RuntimeError:\n                continue\n            rr = {\'conformer\': row.conformer, \'rigid_score\': row.rigid_score, \'refined_score\': float(re.findall(r\'Estimated Free Energy of Binding\\s*:\\s*(-?\\d+\\.?\\d*)\', txt)[-1]),\n                  \'refined_pdbqt\': str(refined)}\n            if reference is not None:\n                rr[\'refined_rmsd_A\'] = reference.rmsd(_to_template_order(o[\'atoms\'], _lines(refined), o[\'conf_heavy\']))\n            refined_rows.append(rr)\n        if not refined_rows:\n            shutil.rmtree(d, ignore_errors=True)\n            return {\'status\': \'FAILED_REFINEMENT\', \'tag\': tag, \'seed\': seed}\n        refined_df = pd.DataFrame(refined_rows).sort_values(\'refined_score\').reset_index(drop=True)\n        best = refined_df.iloc[0]\n        best_pose = self.workdir / f\'{tag}_s{seed}_best.pdbqt\'\n        shutil.copy2(best.refined_pdbqt, best_pose)\n        result = {\'status\': \'DONE\', \'tag\': tag, \'seed\': seed, \'n_conformers\': len(cids), \'n_rigid_docked\': len(ens), \'best_rigid_score\': float(ens.rigid_score.min()),\n                  \'best_refined_score\': float(best.refined_score), \'best_conformer\': int(best.conformer), \'best_pose_pdbqt\': str(best_pose),\n                  \'refined\': refined_df.drop(columns=\'refined_pdbqt\'), \'all_conformers\': ens}\n        if reference is not None:\n            result[\'top_rmsd_A\'] = float(best.refined_rmsd_A)\n            result[\'best_rigid_rmsd_A\'] = float(ens.rigid_rmsd_A.min())\n        if not keep_intermediate:\n            shutil.rmtree(d, ignore_errors=True)\n        return result\n', encoding='utf-8')
import importlib, ace_ensemble_dock as ed; importlib.reload(ed)
settings_file = PATHS['REDOCKING_ANALYSIS'] / '1o86_redocking_validated_settings.json'
USE_ENSEMBLE_PROTOCOL = settings_file.exists()
if USE_ENSEMBLE_PROTOCOL:
    settings = json.loads(settings_file.read_text())
    ENS_PROTOCOL = dict(n_conformers=settings['conformers_per_seed'], rigid_exhaustiveness=settings['rigid_exhaustiveness'], refine_top_k=settings['refine_top_k'])
    SP_SEEDS = [settings['seeds'][0]]        # one seed per compound (a single-seed variant of the 5b protocol); use settings['seeds'] to match 5b exactly at 3x the time
    SEEDS, PROTOCOL_NAME = SP_SEEDS, ed.PROTOCOL_NAME
    N_ACTIVES, N_INACTIVES = min(N_ACTIVES, 20), min(N_INACTIVES, 20)
    print(f'Protocol: conformer-ensemble {ENS_PROTOCOL}, seeds {SEEDS} (slow: minutes per compound)')
else:
    meta_file = DOCK_REPORTS / 'docking_metadata.json'
    meta = json.loads(meta_file.read_text()) if meta_file.exists() else {}
    SEEDS = meta.get('seeds', [7, 42, 2026]); EXHAUSTIVENESS = meta.get('exhaustiveness', 16); NUM_MODES = meta.get('num_modes', 10)
    SCREEN_SEEDS = None              # set e.g. [7] for a quicker (but different-protocol) first look
    if SCREEN_SEEDS is not None: SEEDS = SCREEN_SEEDS
    PROTOCOL_NAME = 'flexible_vina'
    print(f'Protocol: flexible Vina, seeds {SEEDS}, exhaustiveness {EXHAUSTIVENESS}, num_modes {NUM_MODES}' + ('' if meta else '  (defaults; notebook 5 has not run yet)'))


In [ ]:
# 1. Select measured actives and (size-matched) measured inactives
train = pd.read_csv(PROCESSED / 'chembl1808_ic50_clean.csv')
train['heavy_atoms'] = [Chem.MolFromSmiles(s).GetNumHeavyAtoms() for s in train.canonical_smiles]
act_pool = train[(train.ic50_nM <= ACTIVE_MAX_NM) & ~train.ic50_censored.astype(bool)]
ina_pool = train[train.ic50_nM >= INACTIVE_MIN_NM]
print(f'Pools: {len(act_pool)} measured actives (IC50 <= {ACTIVE_MAX_NM:g} nM), {len(ina_pool)} inactives (IC50 >= {INACTIVE_MIN_NM:g} nM; {int(ina_pool.ic50_censored.sum())} censored)')
rng = np.random.RandomState(RANDOM_STATE)

def capped_sample(df, n):
    counts, out = {}, []
    for i in rng.permutation(len(df)):
        sc = df.scaffold.iloc[i]
        if counts.get(sc, 0) < MAX_PER_SCAFFOLD: counts[sc] = counts.get(sc, 0) + 1; out.append(i)
        if len(out) == n: break
    return df.iloc[out]
actives = capped_sample(act_pool, N_ACTIVES)

if SIZE_MATCHED:
    used, counts, picks, unmatched = set(), {}, [], 0
    for ha in sorted(actives.heavy_atoms, reverse=True):
        got = None
        for tol in (SIZE_TOLERANCE, 2 * SIZE_TOLERANCE):
            cand_idx = [i for i in range(len(ina_pool)) if i not in used and abs(ina_pool.heavy_atoms.iloc[i] - ha) <= tol and counts.get(ina_pool.scaffold.iloc[i], 0) < MAX_PER_SCAFFOLD]
            if cand_idx: got = rng.choice(cand_idx); break
        if got is None: unmatched += 1; continue
        used.add(got); counts[ina_pool.scaffold.iloc[got]] = counts.get(ina_pool.scaffold.iloc[got], 0) + 1; picks.append(got)
    inactives = ina_pool.iloc[picks]
    if unmatched: print(f'{unmatched} actives found no size-matched inactive; the design is slightly unbalanced.')
else:
    inactives = capped_sample(ina_pool, N_INACTIVES)
if len(actives) < N_ACTIVES or len(inactives) < N_INACTIVES: print(f'NOTE: requested {N_ACTIVES}/{N_INACTIVES}; got {len(actives)} actives / {len(inactives)} inactives.')
if len(actives) < 8 or len(inactives) < 8: raise RuntimeError('Too few compounds for a meaningful test (need at least 8 per class).')
dock_set = pd.concat([actives.assign(label=1), inactives.assign(label=0)], ignore_index=True)
dock_set['compound_id'] = [f"SP_{'A' if l else 'I'}_{i:03d}_{hashlib.sha1(s.encode()).hexdigest()[:6]}" for i, (l, s) in enumerate(zip(dock_set.label, dock_set.canonical_smiles))]
dock_set.to_csv(BASE / 'screening_power_compounds.csv', index=False)
print(f'Heavy atoms: actives median {actives.heavy_atoms.median():.0f}, inactives median {inactives.heavy_atoms.median():.0f}  ({len(actives)} actives, {len(inactives)} inactives)')


In [ ]:
# 2. Dock every compound with the candidate-docking protocol (resumable)
mode = 'ensemble' if USE_ENSEMBLE_PROTOCOL else 'flexible'
ckpt_file = BASE / f'screening_power_checkpoint_{mode}.csv'
ckpt = pd.read_csv(ckpt_file) if ckpt_file.exists() else pd.DataFrame(columns=['compound_id', 'seed', 'status', 'affinity_kcal_mol'])
records = ckpt[ckpt.status.eq('DONE')].to_dict('records')
t0 = time.time(); n_done0 = len(records)

if USE_ENSEMBLE_PROTOCOL:
    docker = ed.EnsembleDocker(receptor_pdbqt, box, VINA, MEEKO_LIGAND, BASE / 'ensemble', cpu=DOCKING_CPU)
    n_total = len(dock_set) * len(SEEDS); print(f'{n_total} ensemble dockings in total ({n_done0} already done).')
    for r in dock_set.itertuples():
        for seed in SEEDS:
            if any(x['compound_id'] == r.compound_id and x['seed'] == seed for x in records): continue
            out = docker.dock(r.canonical_smiles, r.compound_id, seed, **ENS_PROTOCOL)
            records.append({'compound_id': r.compound_id, 'seed': seed, 'status': out['status'], 'affinity_kcal_mol': out.get('best_refined_score', np.nan)})
            pd.DataFrame(records).to_csv(ckpt_file, index=False)
            n_new = len(records) - n_done0; per = (time.time() - t0) / n_new
            print(f"{len(records)}/{n_total} {r.compound_id}: {out['status']}; {per / 60:.1f} min/run; about {(n_total - len(records)) * per / 3600:.1f} h remaining (estimate)")
else:
    def prepare_ligand(cid, smiles):
        mol = Chem.AddHs(Chem.MolFromSmiles(smiles)); params = AllChem.ETKDGv3(); params.randomSeed = RANDOM_STATE
        ok = AllChem.EmbedMolecule(mol, params)
        if ok != 0: params.useRandomCoords = True; ok = AllChem.EmbedMolecule(mol, params)
        if ok != 0: return 'FAILED_EMBED', None
        try: AllChem.MMFFOptimizeMolecule(mol, maxIters=500)
        except Exception: pass
        sdf, pdbqt = BASE / 'ligands' / f'{cid}.sdf', BASE / 'ligands' / f'{cid}.pdbqt'
        w = Chem.SDWriter(str(sdf)); w.write(mol); w.close()
        r = subprocess.run([MEEKO_LIGAND, '-i', str(sdf), '-o', str(pdbqt)], text=True, capture_output=True)
        return ('READY', pdbqt) if r.returncode == 0 and pdbqt.exists() else ('FAILED_MEEKO', None)
    ligands = {r.compound_id: prepare_ligand(r.compound_id, r.canonical_smiles) for r in dock_set.itertuples()}
    print('Ligand preparation:', pd.Series([v[0] for v in ligands.values()]).value_counts().to_dict())
    ready = [c for c, v in ligands.items() if v[0] == 'READY']
    n_total = len(ready) * len(SEEDS); print(f'{n_total} docking runs in total ({n_done0} already done).')
    for cid in ready:
        for seed in SEEDS:
            if any(r['compound_id'] == cid and r['seed'] == seed for r in records): continue
            pose = BASE / 'runs' / f'{cid}_seed_{seed}.pdbqt'
            cmd = [VINA, '--receptor', str(receptor_pdbqt), '--ligand', str(ligands[cid][1]),
                   '--center_x', str(box['center_x']), '--center_y', str(box['center_y']), '--center_z', str(box['center_z']),
                   '--size_x', str(box['size_x']), '--size_y', str(box['size_y']), '--size_z', str(box['size_z']),
                   '--cpu', str(DOCKING_CPU), '--exhaustiveness', str(EXHAUSTIVENESS), '--num_modes', str(NUM_MODES), '--seed', str(seed), '--out', str(pose)]
            r = subprocess.run(cmd, text=True, capture_output=True); pose.with_suffix('.log').write_text(r.stdout + r.stderr)
            aff = next((float(x.split()[3]) for x in pose.read_text(errors='replace').splitlines() if x.startswith('REMARK VINA RESULT')), np.nan) if pose.exists() else np.nan
            records.append({'compound_id': cid, 'seed': seed, 'status': 'DONE' if r.returncode == 0 and aff == aff else 'FAILED', 'affinity_kcal_mol': aff})
            pd.DataFrame(records).to_csv(ckpt_file, index=False)
            n_new = len(records) - n_done0
            if n_new % 5 == 0:
                per = (time.time() - t0) / n_new; print(f'{len(records)}/{n_total} runs; {per:.0f} s/run; about {(n_total - len(records)) * per / 3600:.1f} h remaining (estimate)')
dk = pd.DataFrame(records); dk = dk[(dk.status == 'DONE') & dk.affinity_kcal_mol.notna()]
best = dk.groupby('compound_id').affinity_kcal_mol.min().rename('best_affinity_kcal_mol')
res = dock_set.merge(best, on='compound_id', how='inner')
print(f'{len(res)} of {len(dock_set)} compounds docked successfully ({int(res.label.sum())} actives, {int((res.label == 0).sum())} inactives).')


In [ ]:
# 3. Screening power: AUC with bootstrap CIs, against size baselines and a size-adjusted score
res = ace_config.annotate_zbg(res)
res['vina_score'] = -res.best_affinity_kcal_mol                       # higher = better predicted binding
res['ligand_efficiency'] = res.vina_score / res.heavy_atoms
def size_adjust(df):
    ina = df[df.label == 0]; slope, icpt = np.polyfit(ina.heavy_atoms, ina.vina_score, 1)   # expected score for a non-binder of that size
    return df.vina_score - (icpt + slope * df.heavy_atoms)
res['size_adjusted_score'] = size_adjust(res)
SCORES = {'Vina score (-dG)': 'vina_score', 'Heavy atoms (size baseline)': 'heavy_atoms', 'Ligand efficiency': 'ligand_efficiency',
          'Size-adjusted Vina score': 'size_adjusted_score', 'Zinc-binding group present (binary)': 'has_zbg'}
res['has_zbg'] = res.has_zbg.astype(float)
y = res.label.to_numpy(); pos_idx, neg_idx = np.where(y == 1)[0], np.where(y == 0)[0]
auc = {k: roc_auc_score(y, res[c]) for k, c in SCORES.items()}
rng_b = np.random.RandomState(RANDOM_STATE); boot = {k: [] for k in SCORES}; boot_diff = []
for _ in range(N_BOOTSTRAP):
    ix = np.concatenate([rng_b.choice(pos_idx, len(pos_idx)), rng_b.choice(neg_idx, len(neg_idx))]); b = res.iloc[ix].copy()
    b['size_adjusted_score'] = size_adjust(b)                          # refit the size trend inside every replicate
    for k, c in SCORES.items(): boot[k].append(roc_auc_score(b.label, b[c]))
    boot_diff.append(boot['Vina score (-dG)'][-1] - boot['Heavy atoms (size baseline)'][-1])
ci = {k: (np.percentile(v, 2.5), np.percentile(v, 97.5)) for k, v in boot.items()}; ci_diff = (np.percentile(boot_diff, 2.5), np.percentile(boot_diff, 97.5))
top_n = max(1, int(round(0.2 * len(res)))); top = res.sort_values('vina_score', ascending=False).head(top_n)
ef20 = top.label.mean() / y.mean()
mw_p = mannwhitneyu(res.loc[y == 1, 'vina_score'], res.loc[y == 0, 'vina_score'], alternative='greater').pvalue
table = pd.DataFrame({'AUC': auc, 'CI95_low': {k: v[0] for k, v in ci.items()}, 'CI95_high': {k: v[1] for k, v in ci.items()}}).round(3); display(table)
print(f'AUC(Vina) - AUC(heavy atoms) = {auc["Vina score (-dG)"] - auc["Heavy atoms (size baseline)"]:+.3f} (95% CI {ci_diff[0]:+.3f} to {ci_diff[1]:+.3f}); enrichment factor in the top 20%: {ef20:.2f}; one-sided Mann-Whitney p = {mw_p:.3g}')
print(f'Median Vina score: actives {res.loc[y == 1, "best_affinity_kcal_mol"].median():.2f}, inactives {res.loc[y == 0, "best_affinity_kcal_mol"].median():.2f} kcal/mol')

lo_v, lo_adj = ci['Vina score (-dG)'][0], ci['Size-adjusted Vina score'][0]
if lo_v >= USEFUL_AUC_LOWER_BOUND and lo_adj > 0.5: verdict = 'USEFUL_BEYOND_SIZE'
elif lo_v > 0.5: verdict = 'SEPARATES_NOT_BEYOND_SIZE'
else: verdict = 'NO_EVIDENCE'
text = {'USEFUL_BEYOND_SIZE': 'The score separates actives from inactives and the effect survives size adjustment: docking can be counted as a supporting signal (still not confirmation).',
        'SEPARATES_NOT_BEYOND_SIZE': 'The score separates the classes, but not demonstrably beyond ligand size: report it, do not count it as independent evidence.',
        'NO_EVIDENCE': 'No evidence that the Vina score separates ACE actives from inactives in this setup: do not use docking scores to support eligibility.'}[verdict]
print('\nVERDICT:', verdict, '-', text)


In [ ]:
# 4. Save results and the ROC figure
fig, ax = plt.subplots(figsize=(5.8, 5.4))
for k, c in SCORES.items():
    fpr, tpr, _ = roc_curve(y, res[c]); ax.plot(fpr, tpr, label=f'{k} (AUC {auc[k]:.2f})', lw=2 if k.startswith('Vina') else 1)
ax.plot([0, 1], [0, 1], 'k--', lw=1); ax.set_xlabel('false positive rate'); ax.set_ylabel('true positive rate'); ax.set_title(f'Docking screening power: {int(y.sum())} actives vs {int((y == 0).sum())} inactives', fontsize=10); ax.legend(fontsize=7, loc='lower right')
fig.tight_layout(); fig.savefig(Path(FIG) / 'docking_screening_power_roc.png', dpi=200); plt.show()
res.to_csv(BASE / 'screening_power_results.csv', index=False)
summary = {'verdict': verdict, 'interpretation': text, 'n_actives': int(y.sum()), 'n_inactives': int((y == 0).sum()), 'size_matched': SIZE_MATCHED,
           'auc': {k: round(v, 4) for k, v in auc.items()}, 'ci95': {k: [round(a, 4), round(b, 4)] for k, (a, b) in ci.items()},
           'auc_vina_minus_heavy_atoms': [round(auc['Vina score (-dG)'] - auc['Heavy atoms (size baseline)'], 4), round(ci_diff[0], 4), round(ci_diff[1], 4)],
           'enrichment_factor_top20pct': round(float(ef20), 3), 'mannwhitney_p_one_sided': float(mw_p), 'activity_cutoff_nM': ACTIVE_MAX_NM, 'inactive_min_nM': INACTIVE_MIN_NM,
           'protocol': ({'name': PROTOCOL_NAME, 'seeds': SEEDS, **ENS_PROTOCOL, 'note': 'single-seed variant of the 5b protocol' if SEEDS != settings['seeds'] else 'same seeds as 5b'} if USE_ENSEMBLE_PROTOCOL else {'name': PROTOCOL_NAME, 'seeds': SEEDS, 'exhaustiveness': EXHAUSTIVENESS, 'num_modes': NUM_MODES}),
           'decision_rule': f'USEFUL_BEYOND_SIZE if Vina AUC CI lower bound >= {USEFUL_AUC_LOWER_BOUND} and size-adjusted AUC CI lower bound > 0.5',
           'receptor_sha256': ace_config.sha256_file(receptor_pdbqt) if hasattr(ace_config, 'sha256_file') else None}
(DOCK_REPORTS / 'docking_screening_power_summary.json').write_text(json.dumps(summary, indent=2, default=float))
print('Saved', DOCK_REPORTS / 'docking_screening_power_summary.json')
